# fact_video_snapshot_poc (periodic snapshot)

Grain: one video × one read (`video_id`, `snapshot_ts`). Counts are semi-additive (take the last read of a period); `*_gained` = vs the previous read of the same video (additive, NULL on the first read).

**Incremental:** reads in the window **+** reads of videos whose `video_key` / `channel_key` is still −1 (the video or channel reached the dim later). The gains need the previous read, which can be older than the window, so the LAG runs over every read of those videos and the window filter comes after it.

Task `fact_video_snapshot_poc` of the job "2. gold (poc tables)", after the dims it reads. Keys come from the `*_scd1` dims (same `xxhash64` keys as the live dims); no match → −1. Test table: the live fact and the agg tables are unchanged.

In [ ]:
%run ./_scd

# Set up environment

In [ ]:
for layer in ("silver", "gold"):
    dbutils.widgets.text(layer, f"prod.{layer}")
dbutils.widgets.dropdown("load_type", "incremental", ["incremental", "full"])
dbutils.widgets.text("from_date", "")   # yyyy-MM-dd, Vietnam day, inclusive; empty = yesterday
dbutils.widgets.text("to_date", "")     # exclusive; empty = tomorrow

silver = dbutils.widgets.get("silver")
gold = dbutils.widgets.get("gold")
load_type = dbutils.widgets.get("load_type")
from_date, to_date = get_incremental_dates(dbutils.widgets.get("from_date"), dbutils.widgets.get("to_date"))

# First load of the table = full (poc update_load_type: full once, then incremental)
if not spark.catalog.tableExists(f"{gold}.fact_video_snapshot_poc"):
    load_type = "full"
print(f"load_type = {load_type}, window = [{from_date}, {to_date})")

# Read Source Table

In [ ]:
path = f"{gold}.fact_video_snapshot_poc"
stg_videos = spark.table(f"{silver}.stg_videos")
dim_video = spark.table(f"{gold}.dim_video_scd1").filter("video_key <> -1")
dim_channel = spark.table(f"{gold}.dim_channel_scd1").filter("channel_key <> -1")

## Incremental: videos read in the window + videos with an unresolved key

In [ ]:
in_window = (F.col("snapshot_date") >= from_date) & (F.col("snapshot_date") < to_date)

if load_type == "incremental":
    changed_videos = stg_videos.filter(in_window).select("video_id")
    unresolved_videos = spark.createDataFrame([], "video_id STRING")
    if spark.catalog.tableExists(path):
        unresolved_videos = spark.table(path).filter("video_key = -1 OR channel_key = -1").select("video_id").distinct()
    stg_videos = stg_videos.join(changed_videos.union(unresolved_videos).distinct(), "video_id", "left_semi")

# Transformation

## Previous read of the same video

In [ ]:
from pyspark.sql.window import Window

previous_read = Window.partitionBy("video_id").orderBy("snapshot_ts")
reads = (stg_videos
         .withColumn("prev_view_count", F.lag("view_count").over(previous_read))
         .withColumn("prev_like_count", F.lag("like_count").over(previous_read))
         .withColumn("prev_comment_count", F.lag("comment_count").over(previous_read))
         .withColumn("prev_snapshot_ts", F.lag("snapshot_ts").over(previous_read)))

if load_type == "incremental":   # keep the window's reads, plus every read of an unresolved video
    reads = (reads.filter(in_window)
             .unionByName(reads.join(unresolved_videos, "video_id", "left_semi"))
             .dropDuplicates(["video_id", "snapshot_ts"]))

## Join dims and select columns

In [ ]:
result_df = (
    reads.alias("read")
    .join(dim_video.alias("video"), on=F.col("read.video_id") == F.col("video.video_id"), how="left")
    .join(dim_channel.alias("channel"), on=F.col("read.channel_id") == F.col("channel.channel_id"), how="left")
    .select(
        F.col("read.video_id").cast("string").alias("video_id"),
        F.col("read.snapshot_ts").cast("timestamp").alias("snapshot_ts"),
        F.date_format("read.snapshot_date", "yyyyMMdd").cast("int").alias("snapshot_date_key"),
        F.coalesce(F.col("video.video_key"), F.lit(-1)).cast("bigint").alias("video_key"),
        F.coalesce(F.col("channel.channel_key"), F.lit(-1)).cast("bigint").alias("channel_key"),
        F.col("read.view_count").cast("bigint").alias("view_count"),
        F.col("read.like_count").cast("bigint").alias("like_count"),
        F.col("read.comment_count").cast("bigint").alias("comment_count"),
        (F.col("read.view_count") - F.col("read.prev_view_count")).cast("bigint").alias("views_gained"),
        (F.col("read.like_count") - F.col("read.prev_like_count")).cast("bigint").alias("likes_gained"),
        (F.col("read.comment_count") - F.col("read.prev_comment_count")).cast("bigint").alias("comments_gained"),
        F.datediff(F.col("read.snapshot_date"),
                   F.to_date(F.from_utc_timestamp("read.published_at", TZ))).cast("int").alias("video_age_days"),
        F.col("read.prev_snapshot_ts").isNull().alias("is_first_snapshot"),
    )
)

## Add metadata

In [ ]:
insert_date = datetime.now()
etl_by = "Databricks"
result_df = result_df.withColumn("etl_date", F.lit(insert_date)).withColumn("etl_by", F.lit(etl_by))

## Add checksums

In [ ]:
primary_key = ["video_id", "snapshot_ts"]
result_df = add_checksums(result_df, primary_key)

## Create table

In [ ]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {path} (
        video_id STRING, snapshot_ts TIMESTAMP, snapshot_date_key INT, video_key BIGINT, channel_key BIGINT,
        view_count BIGINT, like_count BIGINT, comment_count BIGINT,
        views_gained BIGINT, likes_gained BIGINT, comments_gained BIGINT, video_age_days INT, is_first_snapshot BOOLEAN,
        etl_date TIMESTAMP, etl_by STRING, primary_checksum STRING, record_checksum STRING
    ) USING DELTA
    CLUSTER BY (snapshot_date_key)
    COMMENT 'Periodic snapshot at read grain (poc upsert method): 1 video x 1 read. Counts semi-additive, gains (vs previous read) additive.'
""")
result_df = result_df.select(*spark.table(path).columns)

# Write to Table

**full**: overwrite (poc `full_refresh`). **incremental**: upsert (poc `upsert`) = MERGE on `primary_checksum`; update only when `record_checksum` changed; insert new rows. Rows are never deleted.

In [ ]:
if load_type == "full":
    full_refresh(result_df, path)
else:
    result_df.createOrReplaceTempView("source")
    spark.sql(f"""
        MERGE INTO {path} AS t
        USING source AS s
        ON t.primary_checksum = s.primary_checksum
        WHEN MATCHED AND t.record_checksum <> s.record_checksum THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """)

## Check: one row per grain

In [ ]:
n, grain = spark.sql(f"SELECT COUNT(*), COUNT(DISTINCT primary_checksum) FROM {path}").first()
assert n == grain, f"{path}: {n - grain} duplicate rows on (video_id, snapshot_ts)"
last = spark.sql(f"DESCRIBE HISTORY {path} LIMIT 1").select("operation", "operationMetrics").first()
print(f"✅ {path}: {n} rows ·", last["operation"], dict(last["operationMetrics"] or {}))